In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split

## データ読み込み

In [2]:
# データセット
df_raw = pd.read_csv("C:\\keiba\\analysis\\input\\dataset\\dataset.csv", encoding="cp932")

C:\Users\ryo\AppData\Local\Temp\ipykernel_13924\1382220423.py:2: DtypeWarning: Columns (2,5,13,14,15,16,17,21,22,33,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("C:\\keiba\\analysis\\input\\dataset\\dataset.csv", encoding="cp932")


## 設定値

In [44]:
# 頭数
target_num_horses = 12
# 特徴量
features = [
    'time_index_pred_from_race_avg', 
    'jockey_top3_rate_recent_100', 
    'position_half_type_win_rate', 
    'composition_pos_win_rate', 
]

## 前処理

In [87]:
df = df_raw.copy()

# finish_rankから目的変数を作成
# 3着以内なら1、それ以外は0
df["is_top3"] = (df["finish_rank"] <= 3).astype(int)

# 指定した頭数のデータのみに絞り込む
df_sub = df[df["num_horses"] == target_num_horses].copy()

print(f"【初期データ】 {target_num_horses}頭立ての総行数: {len(df_sub)}件")

# --- 【設定】欠損のある馬が何頭以上いるレースを除外するか ---
N = 1

# 1. 特徴量リストの中で、1つでも NaN（欠損値）がある行（馬）を特定 (True/False)
df_sub["is_missing_horse"] = df_sub[features].isna().any(axis=1)

# 2. レース(race_id)ごとに、欠損のある馬が何頭いるかをカウント
missing_counts_per_race = df_sub.groupby("race_id")["is_missing_horse"].sum()

# 3. 欠損のある馬が N頭以上いる race_id を特定して抽出
invalid_race_ids = missing_counts_per_race[missing_counts_per_race >= N].index.tolist()

# 4. 条件を満たすレースのみを抽出（ここではまだNaNが含まれる馬が残っています）
df_clean = df_sub[~df_sub["race_id"].isin(invalid_race_ids)].copy()

print(f"【欠損除外後】 残ったレースの総行数: {len(df_clean)}件 (閾値 N = {N})")
print(f"除外されたレース数: {len(invalid_race_ids)}レース")


# --- 【追加】同じレース内の平均値によるNaNの穴埋め処理 ---
# レース(race_id)ごとにグループ化し、各特徴量の「レース内平均値」を計算してNaNを埋める
# ※もし同一レース内の全馬がNaNの場合は平均値が計算できないため、念のため全体の平均値(fillna(df_clean[features].mean()))で最終補完します。
df_clean[features] = (
    df_clean.groupby("race_id")[features]
    .transform(lambda x: x.fillna(x.mean()))
    .fillna(df_clean[features].mean())
)

# 一時的に作成したフラグ列を削除
df_clean = df_clean.drop(columns=["is_missing_horse"])

print(f"【穴埋め完了】 特徴量の欠損数: {df_clean[features].isna().sum().sum()}件")

【初期データ】 12頭立ての総行数: 193524件
【欠損除外後】 残ったレースの総行数: 85572件 (閾値 N = 1)
除外されたレース数: 8996レース
【穴埋め完了】 特徴量の欠損数: 0件


## 学習

In [88]:
# 1レース内の馬がバラバラにならないよう学習データと検証データを分ける
unique_races = df_clean["race_id"].unique()
split_idx = int(len(unique_races) * 0.4)
train_races = unique_races[:split_idx]
test_races = unique_races[split_idx:]
df_train = df_clean[df_clean["race_id"].isin(train_races)].copy()
df_test = df_clean[df_clean["race_id"].isin(test_races)].copy()
# モデル
model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    objective="binary:logistic",
    eval_metric="logloss"
)
# 学習データ
X_train = df_train[features]
y_train = df_train["is_top3"]
# モデルの学習
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

## 予測

In [89]:
X_test = df_test[features]
# 3着以内（1）になる確率を予測
df_test["pred_prob"] = model.predict_proba(X_test)[:, 1]
# 各レース(race_id)内で「pred_prob」が高い順に順位（1, 2, 3...）をつける
# ※同着（同じ確率）がある場合は、method="first" で強制的に一意の順位を割り振ります
df_test["model_rank"] = (
    df_test.groupby("race_id")["pred_prob"]
    .rank(ascending=False, method="first")
    .astype(int)
)
# 1着予想（pred_probがレースで最も高い馬）にフラグを立てる
df_test["is_model_top"] = (df_test["model_rank"] <= 1).astype(int)
# フラグが立った馬（上位N頭）を抽出してソート
df_pred_finish_rank_top1 = df_test[df_test["is_model_top"] == 1].sort_values(["race_id", "model_rank"]).copy()
# 出力
print(f"対象の総馬数: {len(df_pred_finish_rank_top1)}件")

対象の総馬数: 4279件


## style_pred（脚質予想）ごとの検証（単勝）

In [90]:
df_buy = df_pred_finish_rank_top1.copy()
df_buy["is_hit"] = df_buy["finish_rank"] == 1
df_buy["return_money"] = np.where(df_buy["is_hit"], df_buy["odds"] * 100, 0)
# style_pred ごとに集計
summary = (
    df_buy.groupby("style_pred")
    .agg(
        購入レース数=("is_hit", "count"),
        的中数=("is_hit", "sum"),
        総投資額=("is_hit", lambda x: len(x) * 100),
        総払戻金=("return_money", "sum"),
    )
    .reset_index()
)
total = pd.DataFrame({
    "style_pred": ["全体"],
    "購入レース数": [len(df_buy)],
    "的中数": [df_buy["is_hit"].sum()],
    "総投資額": [len(df_buy) * 100],
    "総払戻金": [df_buy["return_money"].sum()],
})
# style_pred別と結合
summary = pd.concat([summary, total], ignore_index=True)
# 的中率と回収率
summary["単勝的中率(%)"] = (summary["的中数"] / summary["購入レース数"] * 100).round(0).astype(int)
summary["単勝回収率(%)"] = (summary["総払戻金"] / summary["総投資額"] * 100).round(0).astype(int)
# 列の並び替え
summary = summary[
    [
        "style_pred",
        "購入レース数",
        "的中数",
        "単勝的中率(%)",
        "単勝回収率(%)",
        "総投資額",
        "総払戻金",
    ]
]
summary["総投資額"] = summary["総投資額"].astype(int)
summary["総払戻金"] = summary["総払戻金"].astype(int)
# 出力
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(summary)
df_buy.to_csv(f"C:\\keiba\\analysis\\output\\record_{target_num_horses}.csv", encoding="cp932", index=False)
summary.to_csv(f"C:\\keiba\\analysis\\output\\summary_{target_num_horses}.csv", encoding="cp932", index=False)


,style_pred,購入レース数,的中数,単勝的中率(%),単勝回収率(%),総投資額,総払戻金
0,先行,942,310,33,86,94200,80970
1,差し,2652,730,28,92,265200,244010
2,追込,367,98,27,172,36700,63170
3,逃げ,317,121,38,76,31700,24230
4,全体,4279,1259,29,96,427900,412380


In [91]:
# =====================================================================
# 8. 各 style_pred（脚質）× 人気（popularity）別の詳細検証
# =====================================================================

# 各脚質（逃げ・先行・差し・追込）のループ処理を行う
# ※実際のデータにある正確な表記に合わせて順序を設定します
styles = ["逃げ", "先行", "差し", "追込"]

for style in styles:
    # 対象の脚質データだけを抽出
    df_style = df_buy[df_buy["style_pred"] == style].copy()

    # もし対象の脚質データが1件もない場合はスキップ
    if df_style.empty:
        print(f"\n⚠️ {style} のデータが存在しないためスキップします。")
        continue

    # 人気（popularity）ごとに集計
    # ※CSVの人気カラム名が 'popularity' であると仮定しています。違う場合は変更してください。
    pop_summary = (
        df_style.groupby("popularity")
        .agg(
            購入レース数=("is_hit", "count"),
            的中数=("is_hit", "sum"),
            総投資額=("is_hit", lambda x: len(x) * 100),
            総払戻金=("return_money", "sum"),
        )
        .reset_index()
    )

    # 的中率と回収率の計算
    pop_summary["単勝的中率(%)"] = (
        pop_summary["的中数"] / pop_summary["購入レース数"] * 100
    ).round(2)
    pop_summary["単勝回収率(%)"] = (
        pop_summary["総払戻金"] / pop_summary["総投資額"] * 100
    ).round(2)

    # ご指定の列順に並び替え
    pop_summary = pop_summary[
        [
            "popularity",
            "購入レース数",
            "的中数",
            "単勝的中率(%)",
            "単勝回収率(%)",
            "総投資額",
            "総払戻金",
        ]
    ]

    # インデックスを「人気」にして見やすくする
    pop_summary = pop_summary.set_index("popularity")

    # # 表示（表示したいときはコメントアウト外す）
    # print("\n" + "=" * 60)
    # print(f"### 🏇 【{style}】の人気別パフォーマンス詳細 ###")
    # print("=" * 60)
    # with pd.option_context("display.max_rows", None, "display.max_columns", None):
    #     display(pop_summary)

## 追込に絞って他の馬券種も検証

#### payout

In [92]:
# 払戻金
df_payout_raw = pd.read_csv("C:\\keiba\\analysis\\input\\payout\\payout_20240101_20241231.csv", encoding="cp932")

In [93]:
# 対象期間
start_date = "2024-01-01"
end_date = "2024-12-31"

In [94]:
# 各馬券の成績をまとめる集計用
dic_result = {}

#### 複勝

In [145]:
df_payout = df_payout_raw.copy()
df_buy = df_test[df_test["model_rank"] <= 1].copy()
df_buy = df_buy[df_buy["race_id"].astype(str).str.len() != 11]  # レースIDが壊れているレコードを除外
df_buy["race_date"] = pd.to_datetime(df_buy["race_date"])
df_buy = df_buy[
    df_buy["race_date"].between(start_date, end_date)
]
df_buy = df_buy[df_buy["style_pred"] == "追込"]
df_buy["is_hit"] = df_buy["finish_rank"] <= 3
df_payout = df_payout[df_payout["bet_type"] == "複勝"]

df_buy["race_id"] = df_buy["race_id"].astype(str)
df_payout["race_id"] = df_payout["race_id"].astype(str)
df_buy["horse_number"] = df_buy["horse_number"].astype(str)
df_payout["horse_number_1"] = df_payout["horse_number_1"].astype(str)
df_buy = df_buy.merge(
    df_payout[["race_id", "horse_number_1", "payout"]],
    left_on=["race_id", "horse_number"],
    right_on=["race_id", "horse_number_1"],
    how="left"
)
df_buy["payout"] = df_buy["payout"].fillna(0)
df_buy["return_money"] = df_buy["payout"].where(df_buy["is_hit"], 0)
# 列削除
df_buy = df_buy.drop(columns=["horse_number_1", "payout"])
# 購入数・的中数・的中率
buy_count = len(df_buy)
hit_count = df_buy["is_hit"].sum()
hit_rate = hit_count / buy_count if buy_count > 0 else 0
# 購入金額・払戻金額・回収率
buy_money = buy_count * 100
return_money = df_buy["return_money"].sum()
return_rate = return_money / buy_money if buy_money > 0 else 0
# 出力
print("===== 複勝 =====")
print(f"購入数: {buy_count:,}点")
print(f"的中数: {hit_count:,}点")
print(f"的中率: {hit_rate:.2%}")
print(f"購入金額: {buy_money:,}円")
print(f"払戻金額: {return_money:,.0f}円")
print(f"回収率: {return_rate:.2%}")
# 合計用
dic_result["複勝"] = {
    "buy_count": buy_count,
    "hit_count": hit_count,
    "buy_money": buy_money,
    "return_money": return_money
}


===== 複勝 =====
購入数: 130点
的中数: 73点
的中率: 56.15%
購入金額: 13,000円
払戻金額: 14,060円
回収率: 108.15%


In [ ]:
# df_buy[["race_id", "horse_number", "is_top3", "pred_prob", "finish_rank", "is_hit", "return_money"]].head(10)
df_buy.loc[
    df_buy["is_hit"] == 1,
    ["race_id", "horse_number", "finish_rank", "is_hit", "return_money"]
].head(10)

,race_id,horse_number,finish_rank,is_hit,return_money
1,202450010304,3,2.0,True,140.0
6,202455012003,5,1.0,True,100.0
10,202455020404,4,2.0,True,1430.0
11,202454020610,7,1.0,True,230.0
14,202455021804,6,2.0,True,130.0
15,202442021912,3,1.0,True,190.0
16,202442022106,3,3.0,True,190.0
17,202454031002,11,3.0,True,570.0
19,202454031010,2,3.0,True,210.0
20,202445031104,6,1.0,True,140.0


#### ワイド

In [ ]:
df_sim = df_test.copy()
df_sim = df_sim[df_sim["race_id"].astype(str).str.len() != 11]
df_sim["race_date"] = pd.to_datetime(df_sim["race_date"])
df_sim = df_sim[
    df_sim["race_date"].between(start_date, end_date)
]
# model_rank 1～3の中に追込が1頭以上いるrace_id
target_race = (
    df_sim[
        (df_sim["model_rank"] == 1) &
        (df_sim["style_pred"] == "追込")
    ]["race_id"]
    .unique()
)
# 対象レースのみ
df_target = df_sim[
    df_sim["race_id"].isin(target_race)
]
# 1着予想馬
df_axis = (
    df_target[
        df_target["model_rank"] == 1
    ][["race_id", "horse_number", "finish_rank"]]
    .rename(
        columns={
            "horse_number": "horse_number_1",
            "finish_rank": "finish_rank_1",
        }
    )
)
# 2～3着予想馬
df_other = (
    df_target[
        df_target["model_rank"].isin([2, 3])
    ][["race_id", "horse_number", "finish_rank"]]
    .rename(
        columns={
            "horse_number": "horse_number_2",
            "finish_rank": "finish_rank_2",
        }
    )
)
# 組み合わせ作成
df_buy = df_axis.merge(
    df_other,
    on="race_id",
    how="inner"
)
# 的中判定（ワイド：両馬3着以内）
df_buy["is_hit"] = (
    (df_buy["finish_rank_1"] <= 3) &
    (df_buy["finish_rank_2"] <= 3)
).astype(int)
# 払戻金
df_payout = df_payout_raw.copy()
df_payout = df_payout[
    df_payout["bet_type"] == "ワイド"
]
df_payout = df_payout[
    (df_payout["horse_number_1"] != "返還") &
    (df_payout["horse_number_2"] != "返還")
]
# 型統一
df_buy["race_id"] = df_buy["race_id"].astype(str)
df_payout["race_id"] = df_payout["race_id"].astype(str)
df_buy["horse_number_1"] = df_buy["horse_number_1"].astype(int)
df_buy["horse_number_2"] = df_buy["horse_number_2"].astype(int)
df_payout["horse_number_1"] = df_payout["horse_number_1"].astype(int)
df_payout["horse_number_2"] = df_payout["horse_number_2"].astype(int)
# 馬番をソート
df_buy["horse_min"] = (
    df_buy[["horse_number_1", "horse_number_2"]]
    .min(axis=1)
)
df_buy["horse_max"] = (
    df_buy[["horse_number_1", "horse_number_2"]]
    .max(axis=1)
)
df_payout["horse_min"] = (
    df_payout[["horse_number_1", "horse_number_2"]]
    .min(axis=1)
)
df_payout["horse_max"] = (
    df_payout[["horse_number_1", "horse_number_2"]]
    .max(axis=1)
)
# 払戻金結合
df_buy = df_buy.merge(
    df_payout[
        [
            "race_id",
            "horse_min",
            "horse_max",
            "payout"
        ]
    ],
    on=[
        "race_id",
        "horse_min",
        "horse_max"
    ],
    how="left"
)
df_buy["payout"] = df_buy["payout"].fillna(0)
# 購入数・的中数・的中率
buy_count = len(df_buy)
hit_count = df_buy["is_hit"].sum()
hit_rate = (
    hit_count / buy_count
    if buy_count > 0
    else 0
)
# 購入金額・払戻金額・回収率
buy_money = buy_count * 100
return_money = df_buy["payout"].sum()
return_rate = (
    return_money / buy_money
    if buy_money > 0
    else 0
)
# 出力
print("===== ワイド（上位3頭に追込含む） =====")
print(f"購入数: {buy_count:,}点")
print(f"的中数: {hit_count:,}点")
print(f"的中率: {hit_rate:.2%}")
print(f"購入金額: {buy_money:,}円")
print(f"払戻金額: {return_money:,.0f}円")
print(f"回収率: {return_rate:.2%}")
# 合計用
dic_result["ワイド"] = {
    "buy_count": buy_count,
    "hit_count": hit_count,
    "buy_money": buy_money,
    "return_money": return_money
}


===== ワイド（上位3頭に追込含む） =====
購入数: 260点
的中数: 57点
的中率: 21.92%
購入金額: 26,000円
払戻金額: 29,410円
回収率: 113.12%


In [134]:
# df_buy[["race_id", "horse_number_1", "horse_number_2", "finish_rank_1", "finish_rank_2", "is_hit", "payout"]].head(10)
df_buy.loc[
    df_buy["is_hit"] == 1,
    ["race_id", "horse_number_1", "horse_number_2", "finish_rank_1", "finish_rank_2", "is_hit", "payout"]
].head(10)

,race_id,horse_number_1,horse_number_2,finish_rank_1,finish_rank_2,is_hit,payout
3,202450010304,3,10,2.0,1.0,1,340.0
31,202442021912,3,6,1.0,2.0,1,4910.0
40,202445031104,6,2,1.0,2.0,1,740.0
48,202448031201,11,4,2.0,1.0,1,170.0
56,202450031901,7,5,1.0,2.0,1,220.0
70,202442032702,6,1,2.0,1.0,1,3230.0
72,202436032910,7,5,2.0,1.0,1,190.0
75,202455033011,5,11,1.0,2.0,1,560.0
97,202450050104,3,10,2.0,1.0,1,1100.0
129,202448053005,2,9,2.0,1.0,1,970.0


#### 馬連

In [131]:
df_sim = df_test.copy()
df_sim = df_sim[df_sim["race_id"].astype(str).str.len() != 11]
df_sim["race_date"] = pd.to_datetime(df_sim["race_date"])
df_sim = df_sim[
    df_sim["race_date"].between(start_date, end_date)
]
# model_rank 1～3の中に追込が1頭以上いるrace_id
target_race = (
    df_sim[
        (df_sim["model_rank"] == 1) &
        (df_sim["style_pred"] == "追込")
    ]["race_id"]
    .unique()
)
# 対象レースのみ
df_target = df_sim[
    df_sim["race_id"].isin(target_race)
]
# 1着予想馬
df_axis = (
    df_target[
        df_target["model_rank"] == 1
    ][["race_id", "horse_number", "finish_rank"]]
    .rename(
        columns={
            "horse_number": "horse_number_1",
            "finish_rank": "finish_rank_1",
        }
    )
)
# 2～3着予想馬
df_other = (
    df_target[
        df_target["model_rank"].isin([2, 3])
    ][["race_id", "horse_number", "finish_rank"]]
    .rename(
        columns={
            "horse_number": "horse_number_2",
            "finish_rank": "finish_rank_2",
        }
    )
)
# 組み合わせ
df_buy = df_axis.merge(
    df_other,
    on="race_id",
    how="inner"
)
# 的中判定（馬連：2頭とも2着以内）
df_buy["is_hit"] = (
    (df_buy["finish_rank_1"] <= 2) &
    (df_buy["finish_rank_2"] <= 2)
).astype(int)
# 払戻金
df_payout = df_payout_raw.copy()
df_payout = df_payout[
    df_payout["bet_type"] == "馬連"
]
df_payout = df_payout[
    (df_payout["horse_number_1"] != "返還") &
    (df_payout["horse_number_2"] != "返還")
]
# 型統一
df_buy["race_id"] = df_buy["race_id"].astype(str)
df_payout["race_id"] = df_payout["race_id"].astype(str)
df_buy["horse_number_1"] = df_buy["horse_number_1"].astype(int)
df_buy["horse_number_2"] = df_buy["horse_number_2"].astype(int)
df_payout["horse_number_1"] = df_payout["horse_number_1"].astype(int)
df_payout["horse_number_2"] = df_payout["horse_number_2"].astype(int)
# 馬連は順不同なので馬番をソート
df_buy["horse_min"] = (
    df_buy[["horse_number_1", "horse_number_2"]]
    .min(axis=1)
)
df_buy["horse_max"] = (
    df_buy[["horse_number_1", "horse_number_2"]]
    .max(axis=1)
)
df_payout["horse_min"] = (
    df_payout[["horse_number_1", "horse_number_2"]]
    .min(axis=1)
)
df_payout["horse_max"] = (
    df_payout[["horse_number_1", "horse_number_2"]]
    .max(axis=1)
)
# 払戻金結合
df_buy = df_buy.merge(
    df_payout[
        [
            "race_id",
            "horse_min",
            "horse_max",
            "payout"
        ]
    ],
    on=[
        "race_id",
        "horse_min",
        "horse_max"
    ],
    how="left"
)
df_buy["payout"] = df_buy["payout"].fillna(0)
# 購入数・的中数・的中率
buy_count = len(df_buy)
hit_count = df_buy["is_hit"].sum()
hit_rate = (
    hit_count / buy_count
    if buy_count > 0
    else 0
)
# 購入金額・払戻金額・回収率
buy_money = buy_count * 100
return_money = df_buy["payout"].sum()
return_rate = (
    return_money / buy_money
    if buy_money > 0
    else 0
)
# 出力
print("===== 馬連（上位3頭に追込含む） =====")
print(f"購入数: {buy_count:,}点")
print(f"的中数: {hit_count:,}点")
print(f"的中率: {hit_rate:.2%}")
print(f"購入金額: {buy_money:,}円")
print(f"払戻金額: {return_money:,.0f}円")
print(f"回収率: {return_rate:.2%}")
# 合計用
dic_result["馬連"] = {
    "buy_count": buy_count,
    "hit_count": hit_count,
    "buy_money": buy_money,
    "return_money": return_money
}

===== 馬連（上位3頭に追込含む） =====
購入数: 260点
的中数: 24点
的中率: 9.23%
購入金額: 26,000円
払戻金額: 32,130円
回収率: 123.58%


In [133]:
# df_buy[["race_id", "horse_number_1", "horse_number_2", "finish_rank_1", "finish_rank_2", "is_hit", "payout"]].head(10)
df_buy.loc[
    df_buy["is_hit"] == 1,
    ["race_id", "horse_number_1", "horse_number_2", "finish_rank_1", "finish_rank_2", "is_hit", "payout"]
].head(10)

,race_id,horse_number_1,horse_number_2,finish_rank_1,finish_rank_2,is_hit,payout
3,202450010304,3,10,2.0,1.0,1,340.0
31,202442021912,3,6,1.0,2.0,1,4910.0
40,202445031104,6,2,1.0,2.0,1,740.0
48,202448031201,11,4,2.0,1.0,1,170.0
56,202450031901,7,5,1.0,2.0,1,220.0
70,202442032702,6,1,2.0,1.0,1,3230.0
72,202436032910,7,5,2.0,1.0,1,190.0
75,202455033011,5,11,1.0,2.0,1,560.0
97,202450050104,3,10,2.0,1.0,1,1100.0
129,202448053005,2,9,2.0,1.0,1,970.0


#### 馬単

In [113]:
df_sim = df_test.copy()
df_sim = df_sim[df_sim["race_id"].astype(str).str.len() != 11]
df_sim["race_date"] = pd.to_datetime(df_sim["race_date"])
df_sim = df_sim[
    df_sim["race_date"].between(start_date, end_date)
]
# 1着予想馬
df_axis = (
    df_sim[
        (df_sim["model_rank"] == 1) &
        (df_sim["style_pred"] == "追込")
    ][["race_id", "horse_number", "finish_rank"]]
    .rename(
        columns={
            "horse_number": "horse_number_1",
            "finish_rank": "finish_rank_1",
        }
    )
)
# 2～3着予想馬
df_other = (
    df_sim[df_sim["model_rank"].isin([2, 3])]
    [["race_id", "horse_number", "finish_rank"]]
    .rename(
        columns={
            "horse_number": "horse_number_2",
            "finish_rank": "finish_rank_2",
        }
    )
)
# マージ
df_buy = df_axis.merge(df_other, on="race_id", how="inner")
# あたり or はずれ
df_buy["is_hit"] = (
    (df_buy["finish_rank_1"] == 1) &
    (df_buy["finish_rank_2"] == 2)
).astype(int)
# 払戻金
df_payout = df_payout_raw.copy()
df_payout = df_payout[df_payout["bet_type"] == "馬単"]
df_payout = df_payout[df_payout["horse_number_1"] != "返還"]
df_payout = df_payout[df_payout["horse_number_2"] != "返還"]
df_buy["race_id"] = df_buy["race_id"].astype(str)
df_payout["race_id"] = df_payout["race_id"].astype(str)
df_buy["horse_number_1"] = df_buy["horse_number_1"].astype(int)
df_buy["horse_number_2"] = df_buy["horse_number_2"].astype(int)
df_payout["horse_number_1"] = df_payout["horse_number_1"].astype(int)
df_payout["horse_number_2"] = df_payout["horse_number_2"].astype(int)
df_buy = df_buy.merge(
    df_payout[
        ["race_id", "horse_number_1", "horse_number_2", "payout"]
    ],
    on=["race_id", "horse_number_1", "horse_number_2"],
    how="left",
)
df_buy["payout"] = df_buy["payout"].fillna(0)
# 購入数・的中数・的中率
buy_count = len(df_buy)
hit_count = df_buy["is_hit"].sum()
hit_rate = hit_count / buy_count if buy_count > 0 else 0
# 購入金額・払戻金額・回収率
buy_money = buy_count * 100
return_money = df_buy["payout"].sum()
return_rate = return_money / buy_money if buy_money > 0 else 0
# 出力
print("===== 馬単 =====")
print(f"購入数: {buy_count:,}点")
print(f"的中数: {hit_count:,}点")
print(f"的中率: {hit_rate:.2%}")
print(f"購入金額: {buy_money:,}円")
print(f"払戻金額: {return_money:,.0f}円")
print(f"回収率: {return_rate:.2%}")
# 合計用
dic_result["馬単"] = {
    "buy_count": buy_count,
    "hit_count": hit_count,
    "buy_money": buy_money,
    "return_money": return_money
}

===== 馬単 =====
購入数: 260点
的中数: 13点
的中率: 5.00%
購入金額: 26,000円
払戻金額: 30,070円
回収率: 115.65%


In [116]:
df_buy[["race_id", "horse_number_1", "horse_number_2", "finish_rank_1", "finish_rank_2", "is_hit", "payout"]].head(10)

,race_id,horse_number_1,horse_number_2,finish_rank_1,finish_rank_2,is_hit,payout
0,202450010203,3,7,6.0,9.0,0,0.0
1,202450010203,3,8,6.0,8.0,0,0.0
2,202450010304,3,6,2.0,4.0,0,0.0
3,202450010304,3,10,2.0,1.0,0,0.0
4,202450010412,2,11,7.0,1.0,0,0.0
5,202450010412,2,12,7.0,11.0,0,0.0
6,202455010607,8,5,6.0,8.0,0,0.0
7,202455010607,8,9,6.0,2.0,0,0.0
8,202442010905,4,9,5.0,6.0,0,0.0
9,202442010905,4,12,5.0,4.0,0,0.0


#### 三連複

In [140]:
df_sim = df_test.copy()
df_sim = df_sim[df_sim["race_id"].astype(str).str.len() != 11]
df_sim["race_date"] = pd.to_datetime(df_sim["race_date"])
df_sim = df_sim[
    df_sim["race_date"].between(start_date, end_date)
]
# model_rank 1～3の中に追込が1頭以上いるrace_id
target_race = (
    df_sim[
        (df_sim["model_rank"] == 1) &
        (df_sim["style_pred"] == "追込")
    ]["race_id"]
    .unique()
)
# 対象レースのみ
df_target = df_sim[
    df_sim["race_id"].isin(target_race)
]
# 1着予想馬
df_axis = (
    df_target[
        df_target["model_rank"] == 1
    ][
        [
            "race_id",
            "horse_number",
            "finish_rank"
        ]
    ]
    .rename(
        columns={
            "horse_number": "horse_number_1",
            "finish_rank": "finish_rank_1",
        }
    )
)
# 2着・3着予想馬
df_other = (
    df_target[
        df_target["model_rank"].isin([2, 3])
    ][
        [
            "race_id",
            "horse_number",
            "finish_rank",
            "model_rank"
        ]
    ]
)
# model_rank=2,3を横持ち
df_other = df_other.pivot(
    index="race_id",
    columns="model_rank",
    values=[
        "horse_number",
        "finish_rank"
    ]
)
df_other.columns = [
    f"{col[0]}_{col[1]}"
    for col in df_other.columns
]
df_other = df_other.reset_index()
# 軸馬と相手2頭を結合
df_buy = df_axis.merge(
    df_other,
    on="race_id",
    how="inner"
)
# 的中判定（三連複：3頭すべて3着以内）
df_buy["is_hit"] = (
    (df_buy["finish_rank_1"] <= 3) &
    (df_buy["finish_rank_2"] <= 3) &
    (df_buy["finish_rank_3"] <= 3)
).astype(int)
# 払戻金
df_payout = df_payout_raw.copy()
df_payout = df_payout[
    df_payout["bet_type"] == "3連複"
]

df_payout = df_payout[
    (df_payout["horse_number_1"] != "返還") &
    (df_payout["horse_number_2"] != "返還") &
    (df_payout["horse_number_3"] != "返還")
]
# 型統一
df_buy["race_id"] = df_buy["race_id"].astype(str)
df_payout["race_id"] = df_payout["race_id"].astype(str)
for col in [
    "horse_number_1",
    "horse_number_2",
    "horse_number_3"
]:
    df_buy[col] = df_buy[col].astype(int)
    df_payout[col] = df_payout[col].astype(int)
# 三連複は順不同なのでソートキー作成
df_buy[
    [
        "horse_min",
        "horse_mid",
        "horse_max"
    ]
] = np.sort(
    df_buy[
        [
            "horse_number_1",
            "horse_number_2",
            "horse_number_3"
        ]
    ],
    axis=1
)
df_payout[
    [
        "horse_min",
        "horse_mid",
        "horse_max"
    ]
] = np.sort(
    df_payout[
        [
            "horse_number_1",
            "horse_number_2",
            "horse_number_3"
        ]
    ],
    axis=1
)
# 払戻金結合
df_buy = df_buy.merge(
    df_payout[
        [
            "race_id",
            "horse_min",
            "horse_mid",
            "horse_max",
            "payout"
        ]
    ],
    on=[
        "race_id",
        "horse_min",
        "horse_mid",
        "horse_max"
    ],
    how="left"
)
df_buy["payout"] = df_buy["payout"].fillna(0)
# 集計
buy_count = len(df_buy)
hit_count = df_buy["is_hit"].sum()
hit_rate = (
    hit_count / buy_count
    if buy_count > 0
    else 0
)
buy_money = buy_count * 100
return_money = df_buy["payout"].sum()
return_rate = (
    return_money / buy_money
    if buy_money > 0
    else 0
)
# 出力
print("===== 三連複（上位3頭に追込含む） =====")
print(f"購入数: {buy_count:,}点")
print(f"的中数: {hit_count:,}点")
print(f"的中率: {hit_rate:.2%}")
print(f"購入金額: {buy_money:,}円")
print(f"払戻金額: {return_money:,.0f}円")
print(f"回収率: {return_rate:.2%}")
# 合計用
dic_result["三連複"] = {
    "buy_count": buy_count,
    "hit_count": hit_count,
    "buy_money": buy_money,
    "return_money": return_money
}

===== 三連複（上位3頭に追込含む） =====
購入数: 130点
的中数: 8点
的中率: 6.15%
購入金額: 13,000円
払戻金額: 14,310円
回収率: 110.08%


In [141]:
# df_buy[["race_id", "horse_number_1", "horse_number_2", "horse_number_3", "finish_rank_1", "finish_rank_2", "finish_rank_3", "is_hit", "payout"]].head(10)
df_buy.loc[
    df_buy["is_hit"] == 1,
    ["race_id", "horse_number_1", "horse_number_2", "horse_number_3", "finish_rank_1", "finish_rank_2", "finish_rank_3", "is_hit", "payout"]
].head(10)

,race_id,horse_number_1,horse_number_2,horse_number_3,finish_rank_1,finish_rank_2,finish_rank_3,is_hit,payout
20,202445031104,6,2,12,1.0,2.0,3.0,1,980.0
39,202436040811,3,10,2,3.0,2.0,1.0,1,4300.0
74,202448062701,4,7,6,1.0,2.0,3.0,1,850.0
78,202454070605,10,3,6,2.0,1.0,3.0,1,1410.0
98,202448082202,1,3,8,1.0,3.0,2.0,1,960.0
124,202450120304,12,6,10,2.0,1.0,3.0,1,1560.0
125,202448120306,7,11,2,2.0,1.0,3.0,1,1280.0
128,202445121212,3,8,12,1.0,2.0,3.0,1,2970.0


#### 三連単

In [119]:
df_sim = df_test.copy()
df_sim = df_sim[df_sim["race_id"].astype(str).str.len() != 11]
df_sim["race_date"] = pd.to_datetime(df_sim["race_date"])
df_sim = df_sim[
    df_sim["race_date"].between(start_date, end_date)
]
# 1着予想馬
df_axis = (
    df_sim[
        (df_sim["model_rank"] == 1) &
        (df_sim["style_pred"] == "追込")
    ][["race_id", "horse_number", "finish_rank"]]
    .rename(
        columns={
            "horse_number": "horse_number_1",
            "finish_rank": "finish_rank_1",
        }
    )
)
# 2着予想馬
df_second = (
    df_sim[
        df_sim["model_rank"] == 2
    ][["race_id", "horse_number", "finish_rank"]]
    .rename(
        columns={
            "horse_number": "horse_number_2",
            "finish_rank": "finish_rank_2",
        }
    )
)
# 3着予想馬
df_third = (
    df_sim[
        df_sim["model_rank"] == 3
    ][["race_id", "horse_number", "finish_rank"]]
    .rename(
        columns={
            "horse_number": "horse_number_3",
            "finish_rank": "finish_rank_3",
        }
    )
)
# 3頭を結合
df_buy = (
    df_axis
    .merge(df_second, on="race_id", how="inner")
    .merge(df_third, on="race_id", how="inner")
)
# 的中判定（三連単）
df_buy["is_hit"] = (
    (df_buy["finish_rank_1"] == 1) &
    (df_buy["finish_rank_2"] == 2) &
    (df_buy["finish_rank_3"] == 3)
).astype(int)
# 払戻金
df_payout = df_payout_raw.copy()
df_payout = df_payout[df_payout["bet_type"] == "3連単"]
# 返還除外
df_payout = df_payout[
    (df_payout["horse_number_1"] != "返還") &
    (df_payout["horse_number_2"] != "返還") &
    (df_payout["horse_number_3"] != "返還")
]
# 型統一
df_buy["race_id"] = df_buy["race_id"].astype(str)
df_payout["race_id"] = df_payout["race_id"].astype(str)
df_buy["horse_number_1"] = df_buy["horse_number_1"].astype(int)
df_buy["horse_number_2"] = df_buy["horse_number_2"].astype(int)
df_buy["horse_number_3"] = df_buy["horse_number_3"].astype(int)
df_payout["horse_number_1"] = df_payout["horse_number_1"].astype(int)
df_payout["horse_number_2"] = df_payout["horse_number_2"].astype(int)
df_payout["horse_number_3"] = df_payout["horse_number_3"].astype(int)
# 三連単は順番ありなので、そのまま結合
df_buy = df_buy.merge(
    df_payout[
        [
            "race_id",
            "horse_number_1",
            "horse_number_2",
            "horse_number_3",
            "payout"
        ]
    ],
    on=[
        "race_id",
        "horse_number_1",
        "horse_number_2",
        "horse_number_3"
    ],
    how="left"
)
df_buy["payout"] = df_buy["payout"].fillna(0)
# 購入数・的中数・的中率
buy_count = len(df_buy)
hit_count = df_buy["is_hit"].sum()
hit_rate = hit_count / buy_count if buy_count > 0 else 0
# 購入金額・払戻金額・回収率
buy_money = buy_count * 100
return_money = df_buy["payout"].sum()
return_rate = return_money / buy_money if buy_money > 0 else 0
# 出力
print("===== 三連単 =====")
print(f"購入数: {buy_count:,}点")
print(f"的中数: {hit_count:,}点")
print(f"的中率: {hit_rate:.2%}")
print(f"購入金額: {buy_money:,}円")
print(f"払戻金額: {return_money:,.0f}円")
print(f"回収率: {return_rate:.2%}")
# 合計用
dic_result["三連単"] = {
    "buy_count": buy_count,
    "hit_count": hit_count,
    "buy_money": buy_money,
    "return_money": return_money
}

===== 三連単 =====
購入数: 130点
的中数: 3点
的中率: 2.31%
購入金額: 13,000円
払戻金額: 19,130円
回収率: 147.15%


In [ ]:
# df_buy[["race_id", "horse_number_1", "horse_number_2", "horse_number_3", "finish_rank_1", "finish_rank_2", "finish_rank_3", "is_hit", "payout"]].head(10)
df_buy.loc[
    df_buy["is_hit"] == 1,
    ["race_id", "horse_number_1", "horse_number_2", "horse_number_3", "finish_rank_1", "finish_rank_2", "finish_rank_3", "is_hit", "payout"]
].head(10)

,race_id,horse_number_1,horse_number_2,finish_rank_1,finish_rank_2,is_hit,payout
20,202445031104,6,2,1.0,2.0,1,5410.0
74,202448062701,4,7,1.0,2.0,1,2530.0
128,202445121212,3,8,1.0,2.0,1,11190.0


In [121]:
total_buy_count = sum(
    x["buy_count"]
    for x in dic_result.values()
)
total_hit_count = sum(
    x["hit_count"]
    for x in dic_result.values()
)
total_buy_money = sum(
    x["buy_money"]
    for x in dic_result.values()
)
total_return_money = sum(
    x["return_money"]
    for x in dic_result.values()
)
total_hit_rate = (
    total_hit_count / total_buy_count
    if total_buy_count > 0
    else 0
)
total_return_rate = (
    total_return_money / total_buy_money
    if total_buy_money > 0
    else 0
)
print("===== 全券種合計 =====")
print(f"購入数: {total_buy_count:,}点")
print(f"的中数: {total_hit_count:,}点")
print(f"的中率: {total_hit_rate:.2%}")
print(f"購入金額: {total_buy_money:,}円")
print(f"払戻金額: {total_return_money:,.0f}円")
print(f"回収率: {total_return_rate:.2%}")

===== 全券種合計 =====
購入数: 1,170点
的中数: 178点
的中率: 15.21%
購入金額: 117,000円
払戻金額: 139,110円
回収率: 118.90%


## ここから下はあまり使わない

In [148]:
# =====================================================================
# 9. ベースライン検証（モデル関係なしに、各脚質を全買いした場合）
# =====================================================================

# テストデータ全体（df_test）をベースにする
df_all_horses = df_test.copy()

# 【的中判定】実際の着順（finish_rank）が 1 の場合に的中
df_all_horses["is_hit"] = df_all_horses["finish_rank"] == 1

# 【払戻金計算】1点100円で購入したと仮定
df_all_horses["return_money"] = np.where(
    df_all_horses["is_hit"], df_all_horses["odds"] * 100, 0
)

# 全データについて、style_pred ごとに集計（モデルのフラグに関係なく全馬対象）
baseline_summary = (
    df_all_horses.groupby("style_pred")
    .agg(
        総出走数=("is_hit", "count"),  # これが「全買い」したときの購入目数になります
        的中数=("is_hit", "sum"),
        総投資額=("is_hit", lambda x: len(x) * 100),
        総払戻金=("return_money", "sum"),
    )
    .reset_index()
)

# ベタ買い時の 的中率（%）と 回収率（%）を計算
baseline_summary["ベタ買い的中率(%)"] = (
    baseline_summary["的中数"] / baseline_summary["総出走数"] * 100
).round(2)
baseline_summary["ベタ買い回収率(%)"] = (
    baseline_summary["総払戻金"] / baseline_summary["総投資額"] * 100
).round(2)

# 列の並び替え
baseline_summary = baseline_summary[
    [
        "style_pred",
        "総出走数",
        "的中数",
        "ベタ買い的中率(%)",
        "ベタ買い回収率(%)",
        "総投資額",
        "総払戻金",
    ]
]

# 結果の表示
print("\n" + "=" * 50)
print(
    f"### 📊 【ベースライン】モデル関係なし・style_pred別ベタ買い検証 ({target_num_horses}頭) ###"
)
print("=" * 50)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(baseline_summary)


### 📊 【ベースライン】モデル関係なし・style_pred別ベタ買い検証 (12頭) ###


,style_pred,総出走数,的中数,ベタ買い的中率(%),ベタ買い回収率(%),総投資額,総払戻金
0,先行,6988,964,13.80,75.66,698800,528680.0
1,差し,32018,2523,7.88,72.89,3201800,2333660.0
2,追込,10035,407,4.06,66.85,1003500,670820.0
3,逃げ,2290,384,16.77,98.33,229000,225170.0
